# 📈 Financial News Sentiment Analysis + Price Impact
This notebook performs sentiment analysis on financial news and correlates it with stock price movements.

In [ ]:
!pip install yfinance transformers pandas matplotlib seaborn gradio -q

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
import gradio as gr
import datetime

## 📰 Load Financial News Data

In [ ]:
# For demonstration, we will create a small sample of financial news
news_data = pd.DataFrame({
    "date": ["2023-05-01", "2023-05-02", "2023-05-03", "2023-05-04"],
    "headline": [
        "Apple reports record earnings amid supply chain challenges",
        "Federal Reserve raises interest rates by 0.25%",
        "Amazon stock jumps as company beats quarterly expectations",
        "Tech stocks tumble as recession fears grow"
    ]
})
news_data["date"] = pd.to_datetime(news_data["date"])
news_data

## 🔍 Sentiment Analysis with FinBERT

In [ ]:
# Load FinBERT model
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# Apply to headlines
news_data["sentiment"] = news_data["headline"].apply(lambda x: sentiment_pipeline(x)[0]["label"])
news_data

## 📉 Load Stock Price Data

In [ ]:
ticker = "AAPL"
start_date = "2023-04-25"
end_date = "2023-05-10"

stock_data = yf.download(ticker, start=start_date, end=end_date)
stock_data = stock_data.reset_index()[["Date", "Close"]]
stock_data["Date"] = pd.to_datetime(stock_data["Date"])
stock_data

## 🔗 Merge News and Price Data

In [ ]:
merged_df = pd.merge(news_data, stock_data, left_on="date", right_on="Date", how="left")
merged_df = merged_df.drop(columns=["Date"])
merged_df

## 📊 Visualize Sentiment vs Stock Price

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(10,6))
sns.barplot(x="date", y="Close", hue="sentiment", data=merged_df)
plt.title(f"{ticker} Stock Prices vs Sentiment")
plt.xticks(rotation=45)
plt.show()

## 🤖 Gradio Sentiment Tester

In [ ]:
def analyze_sentiment(headline):
    result = sentiment_pipeline(headline)[0]
    return f"Label: {result['label']}, Confidence: {round(result['score'], 2)}"

demo = gr.Interface(fn=analyze_sentiment, 
                    inputs="text", 
                    outputs="text",
                    title="Financial Headline Sentiment Analyzer",
                    description="Enter a financial news headline to predict sentiment using FinBERT")
demo.launch()